# 1 · Deine erste Markov-Kette

[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sem-16161d/GreasyPidginTutorial/blob/main/DE/jupyter/markov-book-de/01_erste_markov_kette.ipynb)

Eine **Markov-Kette** baut eine Folge Schritt für Schritt auf. In jedem Schritt schaut sie nur auf die
letzten paar Symbole (ihr *Gedächtnis*) und wählt das nächste Symbol mit bestimmten Wahrscheinlichkeiten.

In diesem Kapitel gibt es noch keine Musik. Wir nehmen Wetter und Buchstaben, damit du siehst, wie wenig
dafür nötig ist, und lernen die drei GreasyPidgin-Objekte kennen, mit denen wir den ganzen Tag arbeiten:

- `MarkovRule` — eine Zeile des Rezepts: *„nach diesem kommt eines von jenen, mit diesen Gewichten"*
- `MarkovRules` — eine Sammlung von Regeln
- `MarkovChain` — folgt den Regeln und erzeugt eine Folge

Führe die Zellen von oben nach unten mit **Umschalt + Enter** aus. Die erste Zelle installiert alles.

In [ ]:
# Einrichtung -- läuft von selbst (beim ersten Mal etwa 20 Sekunden)
import importlib.util, pathlib, subprocess, sys, urllib.request

GP_REF        = "main"   # welche Version von GreasyPidgin verwendet wird (Git-Branch oder -Tag)
TUTORIAL_REF  = "main"   # welche Version der Hilfsdatei dieses Tutorials verwendet wird
GP_REPO       = "sem-16161d/GreasyPidgin"
TUTORIAL_REPO = "sem-16161d/GreasyPidginTutorial"

if importlib.util.find_spec("GreasyPidgin") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                    f"git+https://github.com/{GP_REPO}.git@{GP_REF}"], check=True)

if not pathlib.Path("workshop_helpers.py").exists():
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/{TUTORIAL_REPO}/{TUTORIAL_REF}/DE/jupyter/markov-book-de/workshop_helpers.py",
        "workshop_helpers.py")

import random
import matplotlib.pyplot as plt
from GreasyPidgin import MarkovChain, MarkovRule, MarkovRules
import workshop_helpers as wh

wh.set_language("de")    # deutsche Meldungen, Beschriftungen und Notennamen
print("Bereit!")

## Eine Kette von Hand: das Wetter

Jede Regel sagt, was auf einen Zustand folgen kann und wie wahrscheinlich jede Möglichkeit ist.
Die Gewichte werden automatisch normiert: `[6, 3, 1]` funktioniert genauso gut wie `[0.6, 0.3, 0.1]`.

In [ ]:
rules = MarkovRules([
    MarkovRule("sonnig",     ["sonnig", "wolkig", "regnerisch"], [0.6, 0.3, 0.1]),
    MarkovRule("wolkig",     ["sonnig", "wolkig", "regnerisch"], [0.3, 0.4, 0.3]),
    MarkovRule("regnerisch", ["sonnig", "wolkig", "regnerisch"], [0.1, 0.4, 0.5]),
])
chain = MarkovChain(rules=rules)
print(rules)

Jetzt lassen wir die Kette 14 Tage laufen. `seed` ist der *Startzustand* der Kette (kein Zufalls-Seed),
`n` ist die Zahl der neuen Schritte. Führe die Zelle mehrmals aus: jedes Mal kommt eine andere Woche heraus.

In [ ]:
tage = chain.getNElements(seed="sonnig", n=14)
print(" -> ".join(tage))

**Probier es aus:** Ändere die Gewichte so, dass der Regen fast nie aufhört. Führe dann die Zelle oben
noch einmal aus und beobachte, wie sich das Wetter verändert.

### Die Regeln sehen

Eine Tabelle mit Wahrscheinlichkeiten ist als Bild leichter zu lesen. Jede Zeile ist der *aktuelle* Zustand,
jede Spalte der *nächste*; je heller, desto wahrscheinlicher.

In [ ]:
wh.plot_transition_matrix(chain, label=str)
plt.show()

## Eine Kette aus Daten lernen

Regeln von Hand zu schreiben wird schnell mühsam. Eine Kette kann ihre Regeln auch **aus einer vorhandenen Folge
lernen**, indem sie zählt, was worauf folgte. Hier ist die Folge ein Wort, ein Buchstabe pro Schritt.

In [ ]:
wort = list("abracadabra")

gelernt = MarkovChain()
gelernt.generateRulesFromList(wort, order=1)   # Ordnung 1: nur der letzte Buchstabe wird gemerkt
print(gelernt.rules)

In [ ]:
neues_wort = gelernt.getNElements(seed="a", n=25)
print("".join(neues_wort))

Nach einem `a` geht das Originalwort mit `b`, `c`, `d` weiter oder (ganz am Ende) springt es zurück zum `a`.
Mehr weiß die Kette nicht — also erfindet sie neue Wörter mit demselben „Geschmack".

### Gedächtnis: Ordnung 2

Bei `order=2` merkt sich die Kette die letzten **zwei** Buchstaben. Dann braucht auch der Startwert zwei Buchstaben.

In [ ]:
gelernt2 = MarkovChain()
gelernt2.generateRulesFromList(wort, order=2)

print("".join(gelernt2.getNElements(seed=("a", "b"), n=25)))

**Probier es aus:**

- Ersetze `"abracadabra"` durch einen eigenen Satz, z. B. `list("der schnelle braune fuchs")`.
- Vergleiche `order=1`, `2`, `3`. Was passiert mit dem Ergebnis, wenn die Ordnung groß wird?
  (Tipp: Mit viel Gedächtnis und wenig Daten hat die Kette bei jedem Schritt nur eine Möglichkeit —
  sie kann nur das Original kopieren.)

Dieser Zielkonflikt zwischen *Kopieren* und *Erfinden* ist die Hauptidee dieses Workshops.
Im nächsten Kapitel machen wir dasselbe mit Noten aus einer MIDI-Datei.